<a href="https://colab.research.google.com/github/yeshaa23/Project-A-Kelompok-4-Pertamina-PBAGenap/blob/main/Week_5_PBA_Preprocessing_Article_Pertamina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing artikel Pertamina
50 artikel dengan pengelompokan manual
https://docs.google.com/spreadsheets/d/1W8cNV7WSMWYG6swvzOR2ZD_CZ9bZnGPyD7a8oR0Vb5E/edit?gid=0#gid=0

Install dan import Library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install openpyxl contractions Sastrawi -q

In [3]:
import pandas as pd
import numpy as np
import re
import html
import contractions
import warnings
warnings.filterwarnings("ignore")

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

## UPLOAD FILE CSV
pakai excel file manual

In [4]:
df = pd.read_excel('/content/drive/MyDrive/ProjectA-PBA/artikel_manual_pertamina.xlsx')

In [5]:
print("Jumlah data awal:", len(df))
print("Kolom awal:", df.columns.tolist())
display(df.head())

Jumlah data awal: 50
Kolom awal: ['Case ID', 'Case', 'Newsroom', 'Tanggal', 'Judul', 'Konten', 'Link', 'Tag', 'Tone']


,Case ID,Case,Newsroom,Tanggal,Judul,Konten,Link,Tag,Tone
0,NaN,NaN,ylki.or.id,2016-06-08 00:00:00,Siaran Pers YLKI : Mendesak PT Pertamina untuk...,Kejadian kecurangan takaran pada SPBU yang ter...,https://ylki.or.id/siaran-pers-ylki-mendesak-p...,Hukum,Negatif
1,NaN,NaN,Kompas.com,6 Juli 2017,Pertamina Kembangkan Program Non-Tunai Pembeli...,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",https://money.kompas.com/read/2017/07/06/14535...,BBM,Netral
2,NaN,NaN,Kompas.com,14 Desember 2017,"Pertamina Tak Masalahkan Nama Pertamini, tapi...","Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",https://otomotif.kompas.com/read/2017/12/14/15...,Bisnis,Positif
3,NaN,NaN,Kompas.com,27 Juli 2017,Pertamina Patra Niaga Akan Bangun SPBU Skala U...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",https://money.kompas.com/read/2017/07/27/17565...,Bisnis,Positif
4,NaN,NaN,Kompas.com,18 Maret 2026,"Gaji Manajer 'Engineering' Chevron Rp 51,25 Ju...","JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",https://money.kompas.com/read/2016/03/18/14152...,Bisnis,Negatif


In [6]:
# rapikan nama kolom kalau ada spasi
df.columns = df.columns.str.strip()

# rename
rename_map = {
    'Judul': 'title',
    'Konten': 'content',
    'Tag': 'tag',
    'Tone': 'sentiment',
    'Tanggal': 'date',
    'Newsroom': 'source',
    'Link': 'url',
    'Case ID': 'case_id',
    'Case': 'case'
}
df = df.rename(columns=rename_map)

In [7]:
print("Kolom setelah rename:")
print(df.columns.tolist())

Kolom setelah rename:
['case_id', 'case', 'source', 'date', 'title', 'content', 'url', 'tag', 'sentiment']


In [8]:
# pilih kolom
cols_to_keep = [col for col in [
    'case_id', 'case', 'source', 'date', 'title', 'content', 'url', 'tag', 'sentiment'
] if col in df.columns]

df = df[cols_to_keep].copy()

print("Kolom yang dipakai:")
print(df.columns.tolist())

Kolom yang dipakai:
['case_id', 'case', 'source', 'date', 'title', 'content', 'url', 'tag', 'sentiment']


## Cleaning data

In [9]:
# drop baris dengan content kosong
df = df.dropna(subset=['content']).copy()

# ubah ke string
for col in ['title', 'content', 'tag', 'sentiment', 'source', 'url']:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

# buang row dengan content kosong setelah strip
df = df[df['content'].str.strip() != ''].copy()

# normalisasi label
if 'tag' in df.columns:
    df['tag'] = df['tag'].astype(str).str.strip().str.title()

if 'sentiment' in df.columns:
    df['sentiment'] = df['sentiment'].astype(str).str.strip().str.title()

print("\nJumlah data setelah cleaning awal:", len(df))


Jumlah data setelah cleaning awal: 50


In [10]:
# hapus duplikat
dup_content = df['content'].duplicated().sum()
print("Duplikat content sebelum hapus:", dup_content)

df = df.drop_duplicates(subset=['content']).copy()

if 'url' in df.columns:
    dup_url = df['url'].duplicated().sum()
    print("Duplikat url sebelum hapus:", dup_url)
    df = df.drop_duplicates(subset=['url']).copy()

print("Jumlah data setelah hapus duplikat:", len(df))

Duplikat content sebelum hapus: 0
Duplikat url sebelum hapus: 0
Jumlah data setelah hapus duplikat: 50


### Stopword dan stemming preparation

In [11]:
stop_factory = StopWordRemoverFactory()
stopwords_id = set(stop_factory.get_stop_words())

# tambah stopword custom biar hasil TF-IDF lebih bersih
custom_stopwords = {
    'kompas', 'cnn', 'detik', 'tempo', 'tribunnews', 'kontan', 'bisnis',
    'com', 'co', 'id', 'news', 'finance', 'money', 'otomotif',
    'jakarta', 'indonesia', 'pt', 'persero', 'tbk'
}
stopwords_id = stopwords_id.union(custom_stopwords)

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

### Hapus boilerplate berita

In [12]:
def remove_news_boilerplate(text: str) -> str:
    text = re.sub(r'\b(kompas|cnn|detik|tempo|tribunnews|kontan|bisnis)\b', ' ', text)
    text = re.sub(r'\b(com|co|id|news|finance|money|otomotif)\b', ' ', text)
    text = re.sub(r'\b(jakarta|indonesia)\b', ' ', text)
    text = re.sub(r'\b(pt|persero|tbk)\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## Sentiment Preprocessing

In [13]:
def sentiment_preprocessing(text: str) -> str:
    text = html.unescape(str(text))
    # normalize quotes
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r"[‘’]", "'", text)
    # collapse duplicate quotes
    text = re.sub(r'""', '"', text)
    text = re.sub(r"''", "'", text)
    # remove literal escaped newline/tab/carriage return
    text = re.sub(r'\\[nrt]+', ' ', text)
    # remove URLs and mentions
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    # expand English contractions
    text = contractions.fix(text)
    # lowercase
    text = text.lower()
    # remove digits if you don't need numbers
    text = re.sub(r'\d+', ' ', text)
    # keep only useful punctuation
    text = re.sub(r"[^\w\s!?%']", " ", text)
    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

## NER and POS Preprocessing

In [14]:
def ner_pos_preprocessing(text: str) -> str:
    text = html.unescape(str(text))

    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r"[‘’]", "'", text)

    text = re.sub(r'""', '"', text)
    text = re.sub(r"''", "'", text)

    text = re.sub(r'\\[nrt]+', ' ', text)

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    text = contractions.fix(text)

    text = re.sub(r'[ \t\r]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

## Stopword removal

In [15]:
def remove_stopwords(text: str) -> str:
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stopwords_id]
    return ' '.join(filtered_tokens)

## Stemming

In [16]:
def stem_text(text: str) -> str:
    return stemmer.stem(text)

## Apply Preprocessing

In [17]:
# versi natural
df['content_clean_nerpos'] = df['content'].apply(ner_pos_preprocessing)
# versi clean dasar
df['content_clean_sentiment'] = df['content'].apply(sentiment_preprocessing)
# hapus boilerplate
df['content_clean_no_boilerplate'] = df['content_clean_sentiment'].apply(remove_news_boilerplate)
# hapus stopword
df['content_clean_no_stopwords'] = df['content_clean_no_boilerplate'].apply(remove_stopwords)
# stemming
df['content_clean_stemmed'] = df['content_clean_no_stopwords'].apply(stem_text)

In [18]:
# hasil final
df['content_final'] = df['content_clean_stemmed']
df['content_clean'] = df['content_clean_sentiment']

In [19]:
# cek hasil
display(df[[
    'content',
    'content_clean',
    'content_final'
]].head())

,content,content_clean,content_final
0,Kejadian kecurangan takaran pada SPBU yang ter...,kejadian kecurangan takaran pada spbu yang ter...,jadi curang takar spbu jadi rempoa ciputat sun...
1,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",pontianak kompas com pt pertamina mensosialisa...,pontianak pertamina sosialisasi program guna u...
2,"Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",nusa dua kompasotomotif keberadaan kios kios p...,nusa kompasotomotif ada kios kios isi bahan ba...
3,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",jakarta kompas com pt pertamina patra niaga se...,pertamina patra niaga anak usaha pertamina ren...
4,"JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",jakarta kompas com situs pencari kerja jobplan...,situs cari kerja jobplanet sebut industri miga...


In [20]:
df['content_length'] = df['content'].astype(str).apply(len)
df['clean_length'] = df['content_clean'].astype(str).apply(len)
df['final_length'] = df['content_final'].astype(str).apply(len)

print("\nJumlah data final:", len(df))
print("Artikel dengan content_clean kosong:", (df['content_clean'].str.strip() == '').sum())
print("Artikel dengan content_final kosong:", (df['content_final'].str.strip() == '').sum())
print("Artikel dengan content < 200 char:", (df['content_length'] < 200).sum())


Jumlah data final: 50
Artikel dengan content_clean kosong: 0
Artikel dengan content_final kosong: 0
Artikel dengan content < 200 char: 0


## Ringkasan Data

In [21]:
print("DISTRIBUSI TAG")
if 'tag' in df.columns:
    print(df['tag'].value_counts(dropna=False))

print("\nDISTRIBUSI SENTIMENT")
if 'sentiment' in df.columns:
    print(df['sentiment'].value_counts(dropna=False))

DISTRIBUSI TAG
tag
Bisnis                  19
Bbm                     10
Hukum                    6
Gangguan Operasional     4
Akademik                 2
Migas                    2
Opini                    2
Ekonomi                  2
                         1
Kecelakaan Kerja         1
Kebakaran                1
Name: count, dtype: int64

DISTRIBUSI SENTIMENT
sentiment
Positif    28
Negatif    19
Netral      3
Name: count, dtype: int64


## Simpan hasil

In [22]:
final_cols = [col for col in [
    'title', 'content', 'tag', 'sentiment',
    'content_clean', 'content_final',
    'source', 'date', 'url',
    'content_length', 'clean_length', 'final_length'
] if col in df.columns]

df_final = df[final_cols].copy()

print("\nKolom akhir:")
print(df_final.columns.tolist())


Kolom akhir:
['title', 'content', 'tag', 'sentiment', 'content_clean', 'content_final', 'source', 'date', 'url', 'content_length', 'clean_length', 'final_length']


In [25]:
output_path = '/content/drive/MyDrive/ProjectA-PBA/artikel_manual_pertamina_preprocessed.csv'
df_final.to_csv(output_path, index=False)

print("File preprocessing disimpan di:", output_path)
print("Ukuran akhir data:", df.shape)
display(df_final.head())

File preprocessing disimpan di: /content/drive/MyDrive/ProjectA-PBA/artikel_manual_pertamina_preprocessed.csv
Ukuran akhir data: (50, 19)


,title,content,tag,sentiment,content_clean,content_final,source,date,url,content_length,clean_length,final_length
0,Siaran Pers YLKI : Mendesak PT Pertamina untuk...,Kejadian kecurangan takaran pada SPBU yang ter...,Hukum,Negatif,kejadian kecurangan takaran pada spbu yang ter...,jadi curang takar spbu jadi rempoa ciputat sun...,ylki.or.id,2016-06-08 00:00:00,https://ylki.or.id/siaran-pers-ylki-mendesak-p...,1500,1450,870
1,Pertamina Kembangkan Program Non-Tunai Pembeli...,"PONTIANAK, KOMPAS.com - PT Pertamina mensosial...",Bbm,Netral,pontianak kompas com pt pertamina mensosialisa...,pontianak pertamina sosialisasi program guna u...,Kompas.com,6 Juli 2017,https://money.kompas.com/read/2017/07/06/14535...,3555,3400,2180
2,"Pertamina Tak Masalahkan Nama Pertamini, tapi...","Nusa Dua, KompasOtomotif - Keberadaan kios-kio...",Bisnis,Positif,nusa dua kompasotomotif keberadaan kios kios p...,nusa kompasotomotif ada kios kios isi bahan ba...,Kompas.com,14 Desember 2017,https://otomotif.kompas.com/read/2017/12/14/15...,1610,1550,1033
3,Pertamina Patra Niaga Akan Bangun SPBU Skala U...,"JAKARTA, KOMPAS.com - PT Pertamina Patra Niaga...",Bisnis,Positif,jakarta kompas com pt pertamina patra niaga se...,pertamina patra niaga anak usaha pertamina ren...,Kompas.com,27 Juli 2017,https://money.kompas.com/read/2017/07/27/17565...,2004,1903,1259
4,"Gaji Manajer 'Engineering' Chevron Rp 51,25 Ju...","JAKARTA, KOMPAS.com - Situs pencari kerja, Job...",Bisnis,Negatif,jakarta kompas com situs pencari kerja jobplan...,situs cari kerja jobplanet sebut industri miga...,Kompas.com,18 Maret 2026,https://money.kompas.com/read/2016/03/18/14152...,1721,1620,1033
